# 01 · Exploración — `data/renta_ine`

**Pipeline:** `01_exploracion` → `02_limpieza` → `03_visualizacion`  
**Objetivo:** entender estructura, encoding, separadores, tipos de columna,
NaN patterns y cardinalidades de los 6 ficheros del INE antes de limpiar.

| Fichero | Alias |
|---------|-------|
| `1.Indicadores de renta media y mediana.csv` | `renta` |
| `2.Distribución por fuente de ingresos.csv` | `fuente_ing` |
| `4.Porcentaje de población ... edad.csv` | `umbrales_edad` |
| `5.Porcentaje de población ... nacionalidad.csv` | `umbrales_nacionalid` |
| `9.Índice de Gini y Distribución de la renta P80P20.csv` | `gini` |
| `10.Indicadores demográficos.csv` | `demografico` |


In [3]:
#=====================================================================
# CELDA 1: Carga PATH
#=====================================================================


# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Estilo visual ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── Rutas (auto-discovery: sube hasta encontrar /data) ────────────────────────
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR   = PROJECT_ROOT / 'data' / 'renta_ine'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'clean'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('DATA existe  :', DATA_DIR.exists())

# ── Ficheros ──────────────────────────────────────────────────────────────────
FILES = {
    'renta':               DATA_DIR / '1.Indicadores de renta media y mediana.csv',
    'fuente_ing':          DATA_DIR / '2.Distribución por fuente de ingresos.csv',
    'umbrales_edad':       DATA_DIR / '4.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y tramos de edad.csv',
    'umbrales_nacionalid': DATA_DIR / '5.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y nacionalidad.csv',
    'gini':                DATA_DIR / '9.Índice de Gini y Distribución de la renta P80P20.csv',
    'demografico':         DATA_DIR / '10.Indicadores demográficos.csv',
}

# ── Constantes ────────────────────────────────────────────────────────────────
ENCODING  = 'utf-8-sig'
SEPARATOR = ';'
NA_VALUES = ['..', '']

# ── Mapa código INE → nombre provincia ───────────────────────────────────────
COD_PROVINCIA = {
    '01':'Álava',        '02':'Albacete',     '03':'Alicante',    '04':'Almería',
    '05':'Ávila',        '06':'Badajoz',      '07':'Baleares',    '08':'Barcelona',
    '09':'Burgos',       '10':'Cáceres',      '11':'Cádiz',       '12':'Castellón',
    '13':'Ciudad Real',  '14':'Córdoba',      '15':'A Coruña',    '16':'Cuenca',
    '17':'Girona',       '18':'Granada',      '19':'Guadalajara', '20':'Gipuzkoa',
    '21':'Huelva',       '22':'Huesca',       '23':'Jaén',        '24':'León',
    '25':'Lleida',       '26':'La Rioja',     '27':'Lugo',        '28':'Madrid',
    '29':'Málaga',       '30':'Murcia',       '31':'Navarra',     '32':'Ourense',
    '33':'Asturias',     '34':'Palencia',     '35':'Las Palmas',  '36':'Pontevedra',
    '37':'Salamanca',    '38':'S.C. Tenerife','39':'Cantabria',   '40':'Segovia',
    '41':'Sevilla',      '42':'Soria',        '43':'Tarragona',   '44':'Teruel',
    '45':'Toledo',       '46':'Valencia',     '47':'Valladolid',  '48':'Bizkaia',
    '49':'Zamora',       '50':'Zaragoza',     '51':'Ceuta',       '52':'Melilla',
}

# ── Verificación final ────────────────────────────────────────────────────────
print('\n🔍 Verificando ficheros:')
all_ok = True
for nombre, path in FILES.items():
    estado = '✅' if path.exists() else '❌ NO ENCONTRADO'
    if not path.exists(): all_ok = False
    print(f'  {estado}  {nombre:<20} → {path.name[:60]}')

print('\n✅ Configuración lista — todos los ficheros encontrados' if all_ok else
      '\n❌ Revisa los ficheros marcados arriba antes de continuar')

PROJECT_ROOT : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow
DATA_DIR     : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\data\renta_ine
OUTPUT_DIR   : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\output\clean
DATA existe  : True

🔍 Verificando ficheros:
  ✅  renta                → 1.Indicadores de renta media y mediana.csv
  ✅  fuente_ing           → 2.Distribución por fuente de ingresos.csv
  ✅  umbrales_edad        → 4.Porcentaje de población con ingresos por unidad de consumo
  ✅  umbrales_nacionalid  → 5.Porcentaje de población con ingresos por unidad de consumo
  ✅  gini                 → 9.Índice de Gini y Distribución de la renta P80P20.csv
  ✅  demografico          → 10.Indicadores demográficos.csv

✅ Configuración lista — todos los ficheros encontrados


In [4]:
#=====================================================================
# CELDA 2: Carga cruda de los 6 CSV y metadata inicial
# (shape, dtypes, nulos, muestra) para orientar la limpieza
#=====================================================================

def metadata_csv(nombre, path):
    df = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING, na_values=NA_VALUES)
    total_celdas = df.shape[0] * df.shape[1]
    nulos_total  = df.isnull().sum().sum()
    print(f"\n{'='*65}")
    print(f"📄 {nombre.upper()}  →  {path.name}")
    print(f"{'='*65}")
    print(f"  Shape        : {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"  Nulos totales: {nulos_total}  ({nulos_total/total_celdas*100:.1f}% del total)")
    print(f"\n  — Nulos por columna —")
    nulos_col = df.isnull().sum()
    nulos_col = nulos_col[nulos_col > 0]
    if nulos_col.empty:
        print("    (ninguna columna con nulos)")
    else:
        for col, n in nulos_col.items():
            print(f"    {col:<50} {n:>6}  ({n/df.shape[0]*100:.1f}%)")
    print(f"\n  — Tipos de dato —")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<50} {str(dtype):<10}")
    print(f"\n  — Primeras 3 filas —")
    print(df.head(3).to_string(index=False))
    return df

# Cargamos y guardamos cada DF en un diccionario para usarlo en celdas siguientes
RAW = {}
for nombre, path in FILES.items():
    RAW[nombre] = metadata_csv(nombre, path)

print(f"\n\n✅ Carga completa — {len(RAW)} DataFrames disponibles en RAW[...]")


📄 RENTA  →  1.Indicadores de renta media y mediana.csv
  Shape        : 3009312 filas × 6 columnas
  Nulos totales: 1532358  (8.5% del total)

  — Nulos por columna —
    Distritos                                          439506  (14.6%)
    Secciones                                          1007424  (33.5%)
    Total                                               85428  (2.8%)

  — Tipos de dato —
    Municipios                                         str       
    Distritos                                          str       
    Secciones                                          str       
    Indicadores de renta media                         str       
    Periodo                                            int64     
    Total                                              str       

  — Primeras 3 filas —
            Municipios Distritos Secciones   Indicadores de renta media  Periodo  Total
01001 Alegría-Dulantzi       NaN       NaN Renta neta media por persona     2023 16.429
01